# Proceso KDD - Sentimind Network

## Minería de Datos y Despliegue de Modelos de Aprendizaje Automático

**Universidad:** ULEAM - Universidad Laica Eloy Alfaro de Manabí

**Proyecto:** Sistema de Análisis de Sentimientos con Zero-Shot Classification

**Fecha:** 2026

---

## Índice

1. [Introducción y Justificación del Problema](#1-introduccion)
2. [Dataset y Fuente de Datos](#2-dataset)
3. [Fase 1: Selección de Datos](#3-seleccion)
4. [Fase 2: Preprocesamiento](#4-preprocesamiento)
5. [Fase 3: Transformación](#5-transformacion)
6. [Fase 4: Minería de Datos](#6-mineria)
7. [Fase 5: Evaluación](#7-evaluacion)
8. [Conclusiones](#8-conclusiones)

---

## 1. Introducción y Justificación del Problema <a name="1-introduccion"></a>

### 1.1 Problema a Resolver

El análisis de sentimientos es una tarea fundamental en el procesamiento del lenguaje natural (NLP) que permite identificar y clasificar las emociones expresadas en textos. Este proyecto aborda el problema de **clasificación multi-emocional de textos en español**, permitiendo a los usuarios de una red social ver automáticamente la categoría emocional de sus publicaciones.

### 1.2 Justificación

1. **Relevancia Social**: Las redes sociales son un canal importante de expresión emocional. Identificar emociones automáticamente puede ayudar a:
   - Detectar contenido que requiere moderación
   - Proporcionar insights sobre el bienestar emocional de comunidades
   - Mejorar la experiencia del usuario con filtros por emoción

2. **Desafío Técnico**: La clasificación multi-emocional es más compleja que el análisis binario (positivo/negativo), ya que:
   - Requiere distinguir entre matices emocionales similares (tristeza vs. nostalgia)
   - Un mismo texto puede expresar múltiples emociones
   - El español tiene sus propias particularidades lingüísticas

3. **Innovación**: Utilizamos **Zero-Shot Classification** que permite:
   - Clasificar sin necesidad de datos de entrenamiento etiquetados
   - Agregar nuevas categorías sin re-entrenar el modelo
   - Trabajar con cualquier idioma soportado por el modelo base

### 1.3 Objetivos

- **General**: Implementar un sistema de clasificación de emociones para textos en español usando técnicas de minería de datos y aprendizaje automático.

- **Específicos**:
  1. Aplicar el proceso KDD completo
  2. Utilizar un modelo de transformers pre-entrenado (XLM-RoBERTa)
  3. Evaluar el modelo con métricas estándar
  4. Integrar el modelo en una aplicación web funcional

---

## 2. Dataset y Fuente de Datos <a name="2-dataset"></a>

### 2.1 Modelo Pre-entrenado

En lugar de un dataset tradicional, este proyecto utiliza **Transfer Learning** con un modelo pre-entrenado:

| Característica | Valor |
|----------------|-------|
| **Modelo** | joeddav/xlm-roberta-large-xnli |
| **Fuente** | Hugging Face Model Hub |
| **Enlace** | https://huggingface.co/joeddav/xlm-roberta-large-xnli |
| **Arquitectura** | XLM-RoBERTa Large |
| **Parámetros** | ~550 millones |
| **Idiomas** | 100+ idiomas (incluido español) |
| **Tarea Original** | Natural Language Inference (NLI) |
| **Adaptación** | Zero-Shot Classification |

### 2.2 Dataset de Entrenamiento Original

El modelo XLM-RoBERTa fue entrenado en:

1. **XNLI Dataset** (Cross-lingual NLI):
   - 392,702 ejemplos de entrenamiento
   - 15 idiomas diferentes
   - Tarea: Inferencia de lenguaje natural

2. **CommonCrawl**:
   - 2.5TB de texto filtrado
   - 100 idiomas
   - Pre-entrenamiento de representaciones

### 2.3 Dataset de Evaluación (Creado para este proyecto)

Para evaluar el modelo en nuestra tarea específica, creamos un dataset de evaluación con:

- **72 textos** etiquetados manualmente
- **18 categorías** emocionales
- **4 ejemplos** por categoría
- Textos en **español** representativos de redes sociales

### 2.4 Justificación del Enfoque Zero-Shot

Elegimos Zero-Shot Classification sobre entrenamiento tradicional porque:

1. **Sin datos etiquetados**: No requerimos un dataset de miles de ejemplos etiquetados
2. **Flexibilidad**: Podemos cambiar las categorías sin re-entrenar
3. **Multilingüe**: El mismo modelo funciona para múltiples idiomas
4. **Estado del arte**: Los modelos de transformers superan a técnicas clásicas en NLP
5. **Transferencia de conocimiento**: Aprovechamos el conocimiento aprendido en tareas similares

---

## 3. Fase 1: Selección de Datos <a name="3-seleccion"></a>

### 3.1 Variables Relevantes

En nuestro caso, la única variable de entrada es:

| Variable | Tipo | Descripción |
|----------|------|-------------|
| `content` | string | Texto del post a clasificar |

Las variables de salida son:

| Variable | Tipo | Descripción |
|----------|------|-------------|
| `primary_category` | string | Emoción principal detectada |
| `primary_confidence` | float | Confianza de la clasificación (0-1) |
| `categories` | list | Lista de emociones detectadas |

### 3.2 Categorías Emocionales Seleccionadas

Definimos **25 categorías** basadas en taxonomías psicológicas de emociones:

```python
TAXONOMY = [
    # Emociones básicas (Ekman)
    "Alegría", "Tristeza", "Enojo", "Miedo", "Sorpresa", "Asco",
    # Emociones sociales
    "Amor", "Odio", "Vergüenza", "Orgullo", "Envidia", "Celos",
    # Tipos de contenido
    "Humor", "Inspiración", "Confesión", "Queja", "Consejo",
    "Pregunta", "Reflexión", "Nostalgia", "Ansiedad", "Frustración",
    # Contenido especial
    "Sarcasmo", "Polémica", "Terror"
]
```

### 3.3 Eliminación de Datos Irrelevantes

Para la entrada del usuario, se aplican filtros básicos:

1. **Longitud mínima**: Textos con menos de 3 caracteres son rechazados
2. **Longitud máxima**: Textos mayores a 1000 caracteres son truncados
3. **Espacios en blanco**: Se eliminan espacios al inicio/final

In [ ]:
# Ejemplo de las categorías definidas
import sys
sys.path.insert(0, '..')

from core.application.ai_service import MiningEngine

print(f"Total de categorías: {len(MiningEngine.TAXONOMY)}")
print("\nCategorías disponibles:")
for i, cat in enumerate(MiningEngine.TAXONOMY, 1):
    print(f"  {i:2d}. {cat}")

---

## 4. Fase 2: Preprocesamiento <a name="4-preprocesamiento"></a>

### 4.1 Preprocesamiento Tradicional vs. Transformers

En NLP tradicional, el preprocesamiento incluye:
- Tokenización
- Eliminación de stopwords
- Lematización/Stemming
- Limpieza de caracteres especiales

**Sin embargo**, los modelos de transformers como XLM-RoBERTa manejan el preprocesamiento internamente de forma más sofisticada:

### 4.2 Tokenización con SentencePiece

El modelo utiliza **SentencePiece** con el algoritmo **BPE (Byte-Pair Encoding)**:

1. **Subword Tokenization**: Divide palabras en subunidades
   - "corriendo" → ["▁corr", "iendo"]
   - Maneja palabras desconocidas efectivamente

2. **Vocabulario**: ~250,000 tokens multilingües

3. **Tokens especiales**:
   - `<s>`: Inicio de secuencia
   - `</s>`: Fin de secuencia
   - `<pad>`: Padding

### 4.3 Manejo de Valores Especiales

| Caso | Manejo |
|------|--------|
| Emojis | Preservados (contienen información emocional) |
| URLs | El tokenizer los maneja como tokens |
| Menciones (@usuario) | Tokenizados normalmente |
| Hashtags | Divididos en subwords |
| Texto vacío | Rechazado antes del modelo |

### 4.4 Normalización

El tokenizer aplica:
- Normalización Unicode (NFD → NFC)
- Conversión a minúsculas (opcional, no usado)
- Eliminación de caracteres de control

In [ ]:
# Demostración de tokenización
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("joeddav/xlm-roberta-large-xnli", use_fast=False)

# Ejemplos de tokenización
ejemplos = [
    "Estoy muy feliz hoy! 😊",
    "Me siento triste y nostálgico",
    "¿Por qué la vida es tan difícil?"
]

print("Demostración de Tokenización:")
print("=" * 50)

for texto in ejemplos:
    tokens = tokenizer.tokenize(texto)
    print(f"\nTexto: '{texto}'")
    print(f"Tokens ({len(tokens)}): {tokens}")

---

## 5. Fase 3: Transformación <a name="5-transformacion"></a>

### 5.1 Embeddings Contextuales

A diferencia de Word2Vec o TF-IDF, XLM-RoBERTa genera **embeddings contextuales**:

- **Dimensión**: 1024 (por token)
- **Contextual**: El mismo token tiene diferentes representaciones según el contexto
- **Multilingüe**: Embeddings comparables entre idiomas

### 5.2 Arquitectura del Modelo

```
Texto → Tokenización → Embeddings → 24 capas Transformer → Clasificación
        SentencePiece   (token +      (self-attention)     (Zero-Shot)
                        position +
                        segment)
```

### 5.3 Zero-Shot Classification

La clasificación Zero-Shot convierte el problema de clasificación en **Natural Language Inference (NLI)**:

1. **Entrada**:
   - Premisa: "Estoy muy feliz hoy"
   - Hipótesis: "Este texto expresa Alegría"

2. **El modelo predice**:
   - Entailment (implica) → Alta probabilidad para esa categoría
   - Contradiction (contradice) → Baja probabilidad
   - Neutral → Probabilidad media

3. **Multi-label**:
   - Se evalúa cada categoría independientemente
   - Se seleccionan las que superen el umbral

### 5.4 Parámetros de Transformación

```python
HYPOTHESIS_TEMPLATE = "Este texto expresa {}"
RELATIVE_THRESHOLD = 0.90  # 90% del score máximo
MAX_EMOTIONS = 3  # Máximo de emociones a retornar
```

In [ ]:
# Demostración de Zero-Shot Classification
from transformers import pipeline

# Clasificador Zero-Shot
classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    device=-1  # CPU
)

# Ejemplo
texto = "Me siento muy triste, extraño los días de mi infancia"
categorias = ["Alegría", "Tristeza", "Nostalgia", "Enojo", "Miedo"]

resultado = classifier(
    texto,
    categorias,
    hypothesis_template="Este texto expresa {}",
    multi_label=True
)

print(f"Texto: '{texto}'")
print("\nResultados:")
for label, score in zip(resultado['labels'], resultado['scores']):
    print(f"  {label:15s}: {score:.2%}")

---

## 6. Fase 4: Minería de Datos <a name="6-mineria"></a>

### 6.1 Selección del Algoritmo

Se eligió **Zero-Shot Classification con XLM-RoBERTa** por las siguientes razones:

| Criterio | Zero-Shot | Naive Bayes | Logistic Regression |
|----------|-----------|-------------|--------------------|
| Datos etiquetados | No requiere | Requiere miles | Requiere miles |
| Precisión en NLP | Alta | Media | Media-Alta |
| Multilingüe | Sí | No | No |
| Nuevas categorías | Sin re-entrenar | Re-entrenar | Re-entrenar |
| Contexto | Comprende contexto | Bag of Words | Bag of Words |
| Emojis/Sarcasmo | Maneja bien | Problemas | Problemas |

### 6.2 Justificación de XLM-RoBERTa

1. **Multilingüe**: Optimizado para 100+ idiomas incluyendo español
2. **XNLI Fine-tuned**: Ya entrenado para tareas de inferencia
3. **Robustez**: RoBERTa mejora BERT con más datos y mejor entrenamiento
4. **Cross-lingual**: Puede transferir conocimiento entre idiomas

### 6.3 Implementación del Motor de Minería

```python
class MiningEngine:
    """Motor de Minería de Texto basado en Transformers."""
    
    # Patrón Singleton para cargar modelo una vez
    _classifier = None
    
    @classmethod
    def analyze(cls, text: str) -> dict:
        """Analiza un texto y retorna emociones detectadas."""
        classifier = cls.get_classifier()
        
        result = classifier(
            text, 
            cls.TAXONOMY, 
            hypothesis_template=cls.HYPOTHESIS_TEMPLATE,
            multi_label=True
        )
        
        # Filtrar por umbral relativo
        # ...
        
        return {
            "categories": detected_categories,
            "primary_category": result['labels'][0],
            "primary_confidence": result['scores'][0]
        }
```

### 6.4 Ajuste de Hiperparámetros

| Hiperparámetro | Valor | Justificación |
|----------------|-------|---------------|
| `multi_label` | True | Permite detectar múltiples emociones |
| `RELATIVE_THRESHOLD` | 0.90 | 90% del score máximo para incluir categoría |
| `MAX_EMOTIONS` | 3 | Máximo de emociones por texto |
| `device` | -1 (CPU) | Compatibilidad sin GPU |

In [ ]:
# Demostración del MiningEngine
import sys
sys.path.insert(0, '..')

from core.application.ai_service import MiningEngine

# Ejemplos de análisis
textos = [
    "Estoy tan feliz, hoy me dieron el trabajo que quería!",
    "Me siento muy triste, mi perro murió ayer",
    "Te odio, eres la peor persona del mundo",
    "Me río para no llorar, perdí todo pero aquí seguimos"
]

print("Demostración del Motor de Minería:")
print("=" * 60)

for texto in textos:
    result = MiningEngine.analyze(texto)
    print(f"\nTexto: '{texto}'")
    print(f"Categoría principal: {result['primary_category']} ({result['primary_confidence']:.1%})")
    print(f"Todas las categorías detectadas:")
    for cat in result['categories']:
        print(f"  - {cat['name']}: {cat['confidence']:.1%}")

---

## 7. Fase 5: Evaluación <a name="7-evaluacion"></a>

### 7.1 Métricas de Evaluación

Para evaluar el modelo, utilizamos métricas estándar de clasificación:

| Métrica | Descripción | Fórmula |
|---------|-------------|---------|
| **Accuracy** | Proporción de predicciones correctas | TP + TN / Total |
| **Precision** | De los predichos positivos, cuántos son correctos | TP / (TP + FP) |
| **Recall** | De los positivos reales, cuántos se detectaron | TP / (TP + FN) |
| **F1-Score** | Media armónica de Precision y Recall | 2 * (P * R) / (P + R) |

### 7.2 Dataset de Evaluación

Creamos un dataset de 72 textos con etiquetas manuales:
- 18 categorías evaluadas
- 4 ejemplos por categoría
- Textos representativos de redes sociales en español

### 7.3 Matriz de Confusión

La matriz de confusión muestra:
- **Diagonal principal**: Predicciones correctas
- **Fuera de diagonal**: Confusiones entre categorías

### 7.4 Ejecución de la Evaluación

El script `evaluate_model.py` genera:
1. Métricas globales (Accuracy, Precision, Recall, F1)
2. Métricas por categoría
3. Matriz de confusión
4. Visualizaciones gráficas

In [ ]:
# Ejecutar evaluación formal
import subprocess
import os

os.chdir('..')
print("Ejecutando evaluación del modelo...")
print("(Esto puede tomar unos minutos)")
print("="*60)

# Nota: Descomentar para ejecutar
# subprocess.run(['python', 'evaluate_model.py'])

### 7.5 Resultados Esperados

Basado en pruebas preliminares, el modelo logra:

| Métrica | Valor Aproximado |
|---------|------------------|
| Accuracy | 70-80% |
| Precision (weighted) | 75-85% |
| Recall (weighted) | 70-80% |
| F1-Score (weighted) | 72-82% |

### 7.6 Análisis de Errores

Las confusiones más comunes ocurren entre:
- **Tristeza ↔ Nostalgia**: Emociones relacionadas
- **Enojo ↔ Frustración**: Expresiones similares
- **Humor ↔ Sarcasmo**: Difíciles de distinguir sin contexto

### 7.7 Visualizaciones Generadas

El script genera las siguientes imágenes en `evaluation_results/`:

1. `confusion_matrix.png` - Matriz de confusión
2. `metrics_by_category.png` - Precision/Recall/F1 por categoría
3. `distribution.png` - Distribución real vs predicho
4. `metrics_summary.png` - Resumen de métricas globales

---

## 8. Conclusiones <a name="8-conclusiones"></a>

### 8.1 Logros del Proyecto

1. **Implementación exitosa** del proceso KDD completo
2. **Uso de tecnología de vanguardia** con Transformers y Zero-Shot
3. **Sistema funcional** con interfaz web y API REST
4. **Evaluación rigurosa** con métricas estándar de ML

### 8.2 Ventajas del Enfoque

- **Sin datos etiquetados**: No necesitamos crear un dataset de entrenamiento
- **Flexibilidad**: Fácil agregar/modificar categorías
- **Multilingüe**: Funciona para múltiples idiomas
- **Estado del arte**: Mejor rendimiento que métodos tradicionales

### 8.3 Limitaciones

- **Recursos computacionales**: Modelo grande (~1.5GB)
- **Latencia**: Primera carga lenta (~30 segundos)
- **Ambigüedad**: Textos sarcásticos son difíciles de clasificar

### 8.4 Trabajo Futuro

1. **Fine-tuning**: Entrenar el modelo específicamente para nuestras categorías
2. **Modelo más ligero**: Usar versión "base" en lugar de "large"
3. **Detección de sarcasmo**: Modelo especializado para casos ambiguos
4. **Análisis temporal**: Tendencias emocionales a lo largo del tiempo

### 8.5 Referencias

1. Conneau, A., et al. (2019). "Unsupervised Cross-lingual Representation Learning at Scale". arXiv:1911.02116
2. Liu, Y., et al. (2019). "RoBERTa: A Robustly Optimized BERT Pretraining Approach". arXiv:1907.11692
3. Yin, W., et al. (2019). "Benchmarking Zero-shot Text Classification". EMNLP 2019
4. Hugging Face Transformers Documentation: https://huggingface.co/docs/transformers